# DLR Geocoding Services

## Goal

Retrieve candidate locations from DLR GeoNames and Photon services.

## What you will do

- Build service URLs.
- Call services through Python.
- Compare candidate tables.
- Save candidates to `outputs/results/geocoder_candidates.csv`.

In [19]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import GEONAMES_BASE_URL, PHOTON_BASE_URL, RESULTS_DIR
from src.data_utils import save_dataframe
import src.geocoder_clients as geocoder_clients


def reload_geocoder_helpers():
    importlib.invalidate_caches()
    importlib.reload(geocoder_clients)
    globals().update(
        {
            "geocode_geonames": geocoder_clients.geocode_geonames,
            "geocode_photon": geocoder_clients.geocode_photon,
            "compare_geocoders": geocoder_clients.compare_geocoders,
        }
    )


reload_geocoder_helpers()


def compact_candidates(df):
    display_columns = [
        "query", "source", "name", "country", "lat", "lon",
        "feature_class", "feature_code", "population",
    ]
    if df.empty:
        return df[[col for col in display_columns if col in df.columns]]
    cols = []
    for col in display_columns:
        if col not in df.columns:
            continue
        if col in {"query", "source", "name", "country", "lat", "lon"}:
            cols.append(col)
        elif df[col].notna().any() and (df[col].astype(str).str.len() > 0).any():
            cols.append(col)
    return df[cols]


## Step 1: Browser URL examples

In [20]:
print('GeoNames:', f'{GEONAMES_BASE_URL}?location=Berlin')
print('Photon:', f'{PHOTON_BASE_URL}?q=Berlin&limit=5')

GeoNames: http://dw-mir-postgis.intra.dlr.de:8091/location?location=Berlin
Photon: http://photon.intra.dlr.de:2322/api/?q=Berlin&limit=5


## Step 2: Query examples

In [21]:
queries = ['Berlin', 'Paris', 'Cambridge', 'Springfield', 'Izmir', 'Bayern', 'Rhine', 'Danube']
queries

['Berlin',
 'Paris',
 'Cambridge',
 'Springfield',
 'Izmir',
 'Bayern',
 'Rhine',
 'Danube']

## Step 3: Call GeoNames defensively

In [22]:
reload_geocoder_helpers()
geonames_berlin = geocode_geonames("Berlin", limit=5)
compact_candidates(geonames_berlin)

,query,source,name,country,lat,lon,feature_class,feature_code,population
0,Berlin,geonames,"Berlin, Stadt (Berlin)",Germany,52.5233,13.41377,A,ADM3,3677472
1,Berlin,geonames,Berlin (Berlino),Germany,52.5233,13.41377,A,ADM4,3677472
2,Berlin,geonames,"Land Berlin (Berlino, BE, 柏林, Berlim, Berlin, ...",Germany,52.5,13.41667,A,ADM1,3677472
3,Berlin,geonames,"Berlin (Berlino, 柏林, Berlim, Berlín, Berlijn)",Germany,52.52437,13.41053,P,PPLC,3426354
4,Berlin,geonames,"Bezirk Neukölln (Bezirk Neukolln, Neukölln, Be...",Germany,52.48333,13.45,A,ADM5,313245


## Step 4: Call Photon defensively

In [23]:
reload_geocoder_helpers()
photon_berlin = geocode_photon("Berlin", limit=5)
compact_candidates(photon_berlin)

,query,source,name,country,lat,lon,feature_code
0,Berlin,photon,Berlin,Deutschland,52.510885,13.398937,city
1,Berlin,photon,Schlacht um Berlin,Deutschland,52.512754,13.381423,house
2,Berlin,photon,Zoo Berlin,Deutschland,52.508449,13.339231,house
3,Berlin,photon,Humboldt-Universität zu Berlin,Deutschland,52.518340,13.392920,house
4,Berlin,photon,Olympiastadion Berlin,Deutschland,52.514585,13.239814,house


## Step 5: Compare candidates

Different geocoders may return different JSON structures and ranking orders.

In [24]:
reload_geocoder_helpers()

candidate_columns = [
    "query", "source", "name", "country", "state", "county",
    "lat", "lon", "feature_class", "feature_code", "population", "raw",
]

frames = []
for query in queries:
    frames.append(compare_geocoders(query, limit=5))

non_empty_frames = [frame for frame in frames if not frame.empty]
candidates = pd.concat(non_empty_frames, ignore_index=True) if non_empty_frames else pd.DataFrame(columns=candidate_columns)
print("Candidates returned:", len(candidates))

Candidates returned: 80


## Step 6: Inspect returned candidates

This notebook does not use mock candidates. If the table is empty, check the configured service URLs and DLR network/VPN access.

In [25]:
if candidates.empty:
    print("No candidates returned by GeoNames or Photon. Check service URLs and DLR network/VPN access.")

compact_candidates(candidates).head(10)

,query,source,name,country,lat,lon,feature_class,feature_code,population
0,Berlin,geonames,"Berlin, Stadt (Berlin)",Germany,52.5233,13.41377,A,ADM3,3677472
1,Berlin,geonames,Berlin (Berlino),Germany,52.5233,13.41377,A,ADM4,3677472
2,Berlin,geonames,"Land Berlin (Berlino, BE, 柏林, Berlim, Berlin, ...",Germany,52.5,13.41667,A,ADM1,3677472
3,Berlin,geonames,"Berlin (Berlino, 柏林, Berlim, Berlín, Berlijn)",Germany,52.52437,13.41053,P,PPLC,3426354
4,Berlin,geonames,"Bezirk Neukölln (Bezirk Neukolln, Neukölln, Be...",Germany,52.48333,13.45,A,ADM5,313245
5,Berlin,photon,Berlin,Deutschland,52.510885,13.398937,None,city,None
6,Berlin,photon,Schlacht um Berlin,Deutschland,52.512754,13.381423,None,house,None
7,Berlin,photon,Zoo Berlin,Deutschland,52.508449,13.339231,None,house,None
8,Berlin,photon,Humboldt-Universität zu Berlin,Deutschland,52.51834,13.39292,None,house,None
9,Berlin,photon,Olympiastadion Berlin,Deutschland,52.514585,13.239814,None,house,None


## Step 7: Save candidate results

These candidates are not final toponym resolution yet.

In [26]:
out = save_dataframe(candidates, RESULTS_DIR / "geocoder_candidates.csv")
print("Saved full candidate table with raw JSON:", out)

Saved full candidate table with raw JSON: /home/hu_xk/Workplace/wawopensearch3_hackathon/module2_geoparsing/outputs/results/geocoder_candidates.csv


## Exercise

Try an ambiguous place name such as `Springfield`, `Cambridge`, or a local place from your own data.

In [27]:
reload_geocoder_helpers()
exercise_query = "Cambridge"
compact_candidates(compare_geocoders(exercise_query, limit=5))

,query,source,name,country,lat,lon,feature_class,feature_code,population
0,Cambridge,geonames,"Boston-Cambridge-Newton, MA-NH Metro Area (Bos...",United States,42.5179,-71.02193,L,RGNE,4550000
1,Cambridge,geonames,Boston-Cambridge-Quincy,United States,42.35347,-71.06094,L,RGNE,4522858
2,Cambridge,geonames,Cambridge (劍橋),United Kingdom,52.2,0.11667,P,PPLA2,145674
3,Cambridge,geonames,Cambridge District,United Kingdom,52.19714,0.12823,A,ADM3,131799
4,Cambridge,geonames,Cambridge (剑桥),Canada,43.3601,-80.31269,P,PPL,129920
5,Cambridge,photon,Cambridge,United Kingdom,52.205531,0.118664,None,city,None
6,Cambridge,photon,Cambridge,United States,42.365635,-71.104002,None,city,None
7,Cambridge,photon,Cambridge,United Kingdom,52.194109,0.137715,None,house,None
8,Cambridge,photon,Cambridge,Canada,43.360054,-80.312302,None,city,None
9,Cambridge,photon,Trinity College (University of Cambridge),United Kingdom,52.20689,0.115113,None,house,None


## Common issues

- Internal endpoints may require DLR network or VPN.
- The notebook display hides mostly empty debug columns and raw JSON.
- The saved CSV keeps the full standardized table, including `raw`, for debugging.
- Missing fields are expected because GeoNames and Photon do not return the same schema.